# 地铁客流入门与误差复算

在本文件所在的 subway flow prediction_for book 目录运行。全网基线只需 NumPy；参考预测由本版已保存的模型运行导出。完成后提交全网 MAE、RMSE，以及一个让两种指标排序不同的原因。

## 1 · 读数据（单元 1）

打开 data/in_15min.csv。预测对象是下一运营时间片各站的进站人数。先猜一行代表什么，再检查形状。

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np

data = Path("data")
flow = np.loadtxt(data / "in_15min.csv", delimiter=",")
print("矩阵（站点，时间片）", flow.shape)
assert flow.shape == (276, 1800)

## 2 · 从已知历史造基线（单元 2）

用前一个已观测时间片预测下一片。若目标索引为 1440，可用的最后一列是哪一列？最后五个工作日能产生多少个站点与时间片组合？

In [ ]:
actual = flow[:, 1440:1800]
previous = flow[:, 1439:1799]
error = previous - actual
mae = np.abs(error).mean()
rmse = np.sqrt((error ** 2).mean())
print(f"{actual.size:,} 组，MAE {mae:.2f} 人，RMSE {rmse:.2f} 人")
assert actual.shape == previous.shape == (276, 360)

<details><summary>核对基线及运营日边界</summary>

99,360 组，MAE 50.67 人，RMSE 101.02 人。1440 的上一列是 1439。每天 05:00 对应的上一列是前一收录工作日 22:45 开始的时间片，测试集有 5 个这样的日首目标。

</details>

## 3 · 从预测节选解释误差（单元 3–4）

读取 data/参考预测_西直门.csv。本版节选固定为留出周首个工作日的西直门，72 行；两个模型用相同的真实值。先核对索引和站名，猜想平方误差与绝对误差会如何给模型排序。

In [ ]:
with (data / "参考预测_西直门.csv").open(encoding="utf-8-sig", newline="") as stream:
    rows = list(csv.DictReader(stream))
info = json.loads((data / "参考预测_说明.json").read_text(encoding="utf-8"))
targets = np.array([int(r["target_index"]) for r in rows])
station = info["station_index_zero_based"]
observed = np.array([float(r["actual"]) for r in rows])
assert len(rows) == info["excerpt_elements"] == 72
np.testing.assert_array_equal(targets, np.arange(1440, 1512))
np.testing.assert_array_equal(observed, flow[station, targets])
print(info["framework"], rows[0]["station"], rows[0]["slot_start"], rows[-1]["slot_start"])

In [ ]:
errors = {}
for field, label in info["fields"].items():
    forecast = np.array([float(r[field]) for r in rows])
    errors[field] = forecast - observed
    mae = np.abs(errors[field]).mean()
    rmse = np.sqrt(np.mean(errors[field] ** 2))
    expected = info["excerpt_metrics"][field]
    np.testing.assert_allclose([mae, rmse], [expected["mae"], expected["rmse"]])
    print(f"{label}：MAE {mae:.2f} 人，RMSE {rmse:.2f} 人")

sq_gain = errors["previous"] ** 2 - errors["dual"] ** 2
abs_cost = np.abs(errors["dual"]) - np.abs(errors["previous"])
for label, values in [("平方误差改善最大", sq_gain), ("绝对误差增量最大", abs_cost)]:
    i = int(np.argmax(values))
    print(label, rows[i]["slot_start"], "真实人数", observed[i], "差值", round(values[i], 2))

## 4 · 写下解释

用单元 4 的两个时间片说明，大误差为何对 RMSE 贡献更大。节选的分母是 72；网页全网表的分母是 99,360。换站、换时段后需要重新计算。进一步练习：把全网基线改成上一收录工作日同时段，保持 1440–1799 的目标不动，再比较两个指标。